# Module 02 · Historical economic headlines and X posts
**Unit:** GDELT's archived **page title**, used as a headline proxy, plus public X posts collected through the independent third-party provider **twitterapi.io**. No article bodies are fetched. GDELT page titles can differ from the editorial H1 headline or include branding; preserve originals and inspect samples.

**News window:** October 2019 onward. GDELT introduced stored page titles on September 22, 2019; this module starts with the first complete month. The economic-only baseline can begin in 2010, but media regressions start only where usable media observations exist. Do not invent 2010–2019 headlines. Partial current months are flagged.

**Workflow:** configure → authenticate → validate schema → dry-run costs → collect a small historical pilot → inspect coverage → collect approved news history → clean and export. Optional X collection is separate, capped and disabled initially. No OpenAI calls occur here.

Selecting fewer returned headlines does NOT necessarily reduce BigQuery bytes scanned: page titles must be extracted from a larger metadata column. Dry runs are essential. BigQuery controls use scan-byte ceilings; twitterapi.io controls use a conservative per-request reserve. Neither is an account-wide budget.

The module creates a **documented source sample**, not a representative sample of all US news or all consumers. National/local source quotas do not establish political balance, audience reach, actual exposure, or causal effects.

**Updated workflow:** first run news and inspect the relevance review; then mark the shared terms reviewed. Only after that can the same vocabulary generate historical X queries. Complete-month X target: 600 posts from October 2019 onward; $10 collection budget. Counts are targets, not guaranteed unique eligible posts.

In [ ]:
# Install only missing connector packages. Do not downgrade Colab's NumPy/SciPy.
import importlib.util, subprocess, sys
try:
    needs_bigquery = importlib.util.find_spec('google.cloud.bigquery') is None
except ModuleNotFoundError:
    needs_bigquery = True
if needs_bigquery:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','google-cloud-bigquery'])
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import json, gzip, hashlib, html, re, unicodedata, os, getpass, time, math
import urllib.request, urllib.parse, urllib.error
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.api_core.exceptions import NotFound, Conflict
try:
    from google.colab import drive, auth
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
# EDIT these two settings.
PROJECT_ROOT=Path('/content/drive/MyDrive/NLP_Headlines_Fresh')
GCP_PROJECT='YOUR_GOOGLE_CLOUD_PROJECT_ID'
LOCATION='US'
NEWS_START='2019-10-01'
# End is EXCLUSIVE, UTC; default omits today's incomplete discovery day.
NEWS_END_EXCLUSIVE=datetime.now(timezone.utc).date().isoformat()
SNAPSHOT_ID=datetime.now(ZoneInfo('America/New_York')).strftime('%Y-%m-%d')
TERMS_VERSION='terms-v3-geo' # change when refining vocabulary; raw news queries remain cached
GEO_FILTER_VERSION='headline-geography-v1'
NEWS_TERMS_REVIEWED=False # inspect news relevance examples before setting True
GKG_TABLE='gdelt-bq.gdeltv2.gkg_partitioned'
BIGQUERY_USD_PER_TIB=6.25  # US on-demand list rate; verify for your billing arrangement
BIGQUERY_BUDGET_USD=10.00 # cumulative module query-compute ceiling, ignoring free allowance
MAX_BYTES_PER_QUERY=2*1024**4  # 2 TiB ceiling, also constrained by remaining budget
PILOT_MONTHS=['2019-10','2020-04','2023-01','2026-07','2026-09']
NEWS_MODE='pilot'         # later change to 'full'
EXECUTE_NEWS=False        # first inspect free dry-run estimates; then set True
MAX_PER_OUTLET_MONTH=50   # deterministic scoring sample; keep full eligible corpus separately
MIN_PANEL_HEADLINES=10
SAMPLING_SEED='headline-study-v1'
# For a fixed sample, freeze these before looking at sentiment results.
# Historical tweets match the headline period. Current partial month is identified separately.
X_START=NEWS_START
X_END_EXCLUSIVE=NEWS_END_EXCLUSIVE
EXECUTE_X=False
X_MODE='pilot'            # 'pilot': 3 windows in 2019/2022/latest complete year; 'history': all
X_TARGET_PER_COMPLETE_MONTH=600
X_EXPECTED_RESULTS_PER_PAGE=20
X_BUDGET_USD=10.00
X_USD_PER_TWEET=.15/1000
X_MIN_REQUEST_USD=.00015
X_RESERVE_RESULTS_PER_REQUEST=100
X_MAX_PAGES_PER_WINDOW=1
X_WINDOW_HOURS=2
X_REQUEST_SPACING_SECONDS=5.1

BASE=PROJECT_ROOT/'02_media_corpus'
RAW=BASE/'raw';JOBDIR=BASE/'bigquery_jobs';XRAW=BASE/'twitterapi_io'
RUN=BASE/'snapshots'/SNAPSHOT_ID/TERMS_VERSION
for folder in [RAW,JOBDIR,XRAW,RUN]:folder.mkdir(parents=True,exist_ok=True)
assert pd.Timestamp(NEWS_START)>=pd.Timestamp('2019-09-23'), 'Stored GKG titles do not support a 2010 start.'
assert pd.Timestamp(NEWS_START)<pd.Timestamp(NEWS_END_EXCLUSIVE)
assert X_TARGET_PER_COMPLETE_MONTH>0 and X_EXPECTED_RESULTS_PER_PAGE>0
assert X_MAX_PAGES_PER_WINDOW==1, 'This 600/month design assumes one page/window; revise sampling design if changing.'
assert GCP_PROJECT!='YOUR_GOOGLE_CLOUD_PROJECT_ID','Enter your billing-enabled Google Cloud project ID.'

def atomic_json(path,data):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2,default=str));tmp.replace(path)

def sha(text):return hashlib.sha256(str(text).encode()).hexdigest()

## 1. Document source selection, search terms and inclusion rules
Sixteen pre-selected outlets span national general news, national business news and local/metro sources in New York, Los Angeles, Chicago and Houston. Selection is pragmatic: English-language US economic coverage, source-type variety and historical availability to be tested. It does not represent all regions or ideologies. Domains and names may change; zeros are coverage findings, not evidence of zero news production.

The source registry includes intended geography and audience information fields. Numerical audience information is **unknown unless sourced**; none is invented. No audience/exposure weights are applied.

To preserve a retrieval denominator, the query returns available titles for selected domains before sentiment filtering. Economic eligibility is screened locally using symmetric favorable/unfavorable keyword groups. This is a keyword-defined economic sample; keyword counts are not final topic labels. Economic stories lacking the terms can be missed. Human/AI eligibility review in the next module must examine US relevance and ambiguity. English is inferred provisionally from selected English-language outlets and translation metadata, not certified by domain alone.

`SEARCH_TERMS` is the single editable vocabulary for both platforms. News uses word/phrase regex matching; X uses OR queries generated from those exact words/phrases, with a separately documented US cue and language/repost filters. Platform search semantics can differ even with the same vocabulary.

In [ ]:
source_rows=[
 ('CBS News','cbsnews.com','National-general','United States'),
 ('CNN','cnn.com','National-general','United States'),
 ('Fox News','foxnews.com','National-general','United States'),
 ('NPR','npr.org','National-general','United States'),
 ('Business Insider','businessinsider.com','National-business','United States'),
 ('CNBC','cnbc.com','National-business','United States'),
 ('Fortune','fortune.com','National-business','United States'),
 ('Yahoo Finance','finance.yahoo.com','National-business','United States'),
 ('ABC13 Houston','abc13.com','Local','Houston, Texas'),
 ('Houston Public Media','houstonpublicmedia.org','Local','Houston, Texas'),
 ('Chicago Tribune','chicagotribune.com','Local','Chicago, Illinois'),
 ('WBEZ','wbez.org','Local','Chicago, Illinois'),
 ('Los Angeles Times','latimes.com','Local','Los Angeles, California'),
 ('LAist','laist.com','Local','Los Angeles, California'),
 ('Gothamist','gothamist.com','Local','New York City, New York'),
 ('New York Post','nypost.com','Local','New York City, New York')]
sources=pd.DataFrame(source_rows,columns=['outlet','domain','source_category','intended_geography'])
sources['source_country']='United States';sources['expected_language']='English'
sources['selection_reason']='Pre-specified national/general, business, or four-metro source coverage; availability tested in pilot'
sources['source_url']='https://'+sources.domain
sources['audience_estimate']=np.nan;sources['audience_reference_date']='';sources['audience_source_url']=''
sources['audience_status']='Unknown; no audience or exposure weighting'
# Edit this ONE vocabulary after reviewing pilot news, before freezing it for X.
# No automatic selection of words based on sentiment or correlation results.
SEARCH_TERMS={
 'prices':['inflation','deflation','disinflation','consumer prices','cost of living','grocery prices',
           'food prices','gas prices','energy prices','price hikes','rising prices','falling prices'],
 'employment':['unemployment','employment','layoff','layoffs','laid off','hiring','job cuts',
               'job gains','job growth','jobs report','labor market','payrolls','job openings'],
 'income':['wages','wage','salaries','household income','personal income','pay raise','pay raises',
           'purchasing power','real income'],
 'growth':['GDP','gross domestic product','recession','economic growth','economic slowdown',
           'economy','economic outlook','productivity','soft landing','hard landing'],
 'housing':['mortgage','mortgages','housing market','housing costs','housing affordability',
            'home prices','rent','rents','renters','eviction','evictions'],
 'household_finance':['consumer confidence','consumer sentiment','consumer spending','household spending',
           'credit card debt','student loan','student loans','debt burden','borrowing costs',
           'interest rates','Federal Reserve','Fed rates']}
SEARCH_GROUPS={group:r'\b(?:'+'|'.join(re.escape(term) for term in terms)+r')\b'
               for group,terms in SEARCH_TERMS.items()}
TERMS_HASH=sha(json.dumps(SEARCH_TERMS,sort_keys=True))
COLLECTION_CONFIG=dict(news_start=NEWS_START,news_end_exclusive=NEWS_END_EXCLUSIVE,gkg_table=GKG_TABLE,
 sources=sources[['outlet','domain','source_category']].to_dict('records'),search_terms=SEARCH_TERMS,search_groups=SEARCH_GROUPS,terms_version=TERMS_VERSION,terms_hash=TERMS_HASH,geo_filter_version=GEO_FILTER_VERSION,
 query_version='gkg-headline-only-v1',sampling_seed=SAMPLING_SEED,max_per_outlet_month=MAX_PER_OUTLET_MONTH,
 min_panel_headlines=MIN_PANEL_HEADLINES,discovery_timestamp_rule='first observation of the selected stored page title',
 x_provider='twitterapi.io; independent third party',x_start=X_START,x_end_exclusive=X_END_EXCLUSIVE,
 x_target_monthly=X_TARGET_PER_COMPLETE_MONTH,x_pages=X_MAX_PAGES_PER_WINDOW,
 x_window_hours=X_WINDOW_HOURS,x_query_type='Latest',x_sampling='deterministic stratified time windows; rotating query groups')
# Fingerprint drives raw-query cache; sampling changes do not require another BigQuery scan.
SOURCE_HASH=sha(json.dumps(sources[['outlet','domain','source_category']].to_dict('records'),sort_keys=True))[:12]
RAWSET=RAW/('sources_'+SOURCE_HASH);RAWSET.mkdir(exist_ok=True)
old=RUN/'collection_config.json'
if old.exists() and json.loads(old.read_text())!=COLLECTION_CONFIG:
    raise ValueError('Snapshot settings differ. Change SNAPSHOT_ID to preserve the earlier version.')
atomic_json(old,COLLECTION_CONFIG)
sources.to_csv(RUN/'source_registry.csv',index=False)
atomic_json(RUN/'search_terms.json',{'terms':SEARCH_TERMS,'regexes':SEARCH_GROUPS,'terms_hash':TERMS_HASH})

## 2. Authenticate and validate the live schema — no data scan
Use your Google Cloud project with BigQuery enabled and billing configured. Colab authentication uses your selected Google account. Schema metadata reads are used to confirm the stored-title column and partition field before constructing SQL.

The query extracts page titles and optional publication metadata **in BigQuery**. It does not download `Extras`, article bodies, GKG tone scores or emotional predictions. Missing titles remain in the retrieval audit. No relevance/sentiment sampling happens inside the SQL.

In [ ]:
try:
    auth.authenticate_user()
except NameError:
    pass # outside Colab, use Application Default Credentials
client=bigquery.Client(project=GCP_PROJECT,location=LOCATION)
table=client.get_table(GKG_TABLE)
fields={f.name:f.field_type for f in table.schema}
EXTRAS=next((x for x in ['Extras','XMLExtras'] if x in fields),None)
required=['DATE','DocumentIdentifier','SourceCommonName','TranslationInfo']
missing=[x for x in required if x not in fields]
if EXTRAS is None or missing: raise ValueError(f'GKG schema changed: missing {missing}; metadata field={EXTRAS}')
if not table.time_partitioning: raise ValueError('Expected time-partitioned table; do not run an unpruned history scan.')
atomic_json(RUN/'gkg_schema.json',{'table':GKG_TABLE,'fields':fields,'extras_column':EXTRAS,
    'partition_field':table.time_partitioning.field,'partition_type':table.time_partitioning.type_})
print('Schema validated. Title source:',EXTRAS,'| no article-body downloads.')

# One scan per month across ALL selected domains; not one scan per outlet.
# Raw title versions are retained so later titles are never backdated to an earlier crawl.
SQL=rf"""
WITH selected AS (
 SELECT CAST(DATE AS STRING) AS gdelt_date_raw,
        DocumentIdentifier AS url,
        LOWER(REGEXP_REPLACE(NET.HOST(DocumentIdentifier),r'^www\.', '')) AS domain,
        REGEXP_EXTRACT({EXTRAS},r'(?is)<PAGE_TITLE>(.*?)</PAGE_TITLE>') AS page_title_raw,
        REGEXP_EXTRACT({EXTRAS},r'<PAGE_PRECISEPUBTIMESTAMP>([0-9]{{14}})</PAGE_PRECISEPUBTIMESTAMP>') AS publication_raw,
        REGEXP_EXTRACT(TranslationInfo,r'srclc:([^;]+)') AS source_language_code
 FROM `{GKG_TABLE}`
 WHERE _PARTITIONTIME >= TIMESTAMP(@start_date)
   AND _PARTITIONTIME < TIMESTAMP(@end_date)
), domain_filtered AS (
 SELECT * FROM selected
 WHERE EXISTS (SELECT 1 FROM UNNEST(@domains) d
               WHERE domain=d OR ENDS_WITH(domain,CONCAT('.',d)))
)
SELECT url,domain,page_title_raw,publication_raw,source_language_code,
       MIN(gdelt_date_raw) AS title_first_seen_raw,
       MAX(gdelt_date_raw) AS title_last_seen_raw,
       COUNT(*) AS gdelt_record_count
FROM domain_filtered
GROUP BY url,domain,page_title_raw,publication_raw,source_language_code
"""
(RUN/'headline_query.sql').write_text(SQL)

def boundaries(month):
    m=pd.Timestamp(month+'-01')
    start=max(m,pd.Timestamp(NEWS_START));end=min(m+pd.offsets.MonthBegin(1),pd.Timestamp(NEWS_END_EXCLUSIVE))
    return start.date().isoformat(),end.date().isoformat()

def query_params(start,end):
    return [bigquery.ScalarQueryParameter('start_date','DATE',start),
            bigquery.ScalarQueryParameter('end_date','DATE',end),
            bigquery.ArrayQueryParameter('domains','STRING',sources.domain.tolist())]

def query_identity(month):
    start,end=boundaries(month)
    fingerprint=sha(json.dumps({'sql':SQL,'start':start,'end':end,'domains':sources.domain.tolist()},sort_keys=True))
    return fingerprint,RAWSET/f'{month}_{fingerprint[:12]}.jsonl.gz'

def estimate_month(month):
    start,end=boundaries(month);fingerprint,path=query_identity(month)
    if start>=end: raise ValueError(f'Empty time interval for {month}')
    job=client.query(SQL,location=LOCATION,job_config=bigquery.QueryJobConfig(
       dry_run=True,use_query_cache=False,query_parameters=query_params(start,end)))
    scanned=int(job.total_bytes_processed or 0)
    return dict(month=month,start=start,end_exclusive=end,estimated_bytes=scanned,
        tib=scanned/1024**4,estimated_list_usd=scanned/1024**4*BIGQUERY_USD_PER_TIB,
        query_fingerprint=fingerprint,raw_path=str(path),already_saved=path.exists())

all_months=pd.period_range(pd.Timestamp(NEWS_START).to_period('M'),
                          (pd.Timestamp(NEWS_END_EXCLUSIVE)-pd.Timedelta(days=1)).to_period('M'),freq='M').astype(str).tolist()
pilot_months=[m for m in PILOT_MONTHS if m in all_months]
if not pilot_months:pilot_months=[all_months[0],all_months[-1]]
selected_months=pilot_months if NEWS_MODE=='pilot' else all_months
plan=pd.DataFrame([estimate_month(m) for m in selected_months])
plan.to_csv(RUN/f'news_{NEWS_MODE}_dry_run.csv',index=False)
display(plan[['month','tib','estimated_list_usd','already_saved']])
print('Estimated list cost for unsaved queries: $',round(plan.loc[~plan.already_saved,'estimated_list_usd'].sum(),2))
print('Free allowances/credits are NOT assumed. LIMIT and smaller output samples do not cap scan costs.')

## 3. Execute only after inspecting costs
Set `EXECUTE_NEWS=True` in configuration and rerun the configuration cell and cells below. Execution mode is not part of the frozen scientific settings. `NEWS_MODE='pilot'` downloads a few separated months to check early/late title availability and source coverage; later select `'full'` for the complete monthly plan.

Each query receives a server-enforced `maximum_bytes_billed`. A persistent job ledger counts actual billed bytes for finished jobs and reserves the maximum for unresolved submissions. Unknown/pending jobs block further collection until inspected, preventing accidental repeated charges after interruption. This ceiling covers this module's query compute only; other projects, query jobs, storage/egress, taxes or different pricing arrangements are outside it. Use one running collector, not parallel notebooks.

**Do not treat a budget-stopped, chronologically incomplete collection as a balanced final corpus.** Coverage exports below mark every missing month. Raise the budget or narrow the pre-specified window after inspecting scan estimates, before sentiment analysis.

In [ ]:
def job_records():
    return [json.loads(p.read_text()) for p in JOBDIR.glob('*.json')]

def reserved_cost():
    return sum(r.get('budget_bytes',0) for r in job_records())/1024**4*BIGQUERY_USD_PER_TIB

def collect_month(plan_row):
    month=plan_row['month'];fingerprint,path=query_identity(month)
    if path.exists():return {'month':month,'status':'cached','path':str(path)}
    jid='headline_'+fingerprint[:40]
    jp=JOBDIR/(jid+'.json')
    previous=json.loads(jp.read_text()) if jp.exists() else None
    if previous:
        try:job=client.get_job(jid,project=GCP_PROJECT,location=LOCATION)
        except NotFound:
            raise RuntimeError(f'{jid}: submission status unknown; inspect Cloud job history before retrying.') from None
        if job.state!='DONE':raise RuntimeError(f'{jid}: still running. Wait and rerun; do not resubmit.')
        if job.error_result:raise RuntimeError(f'{jid}: failed. Inspect its error in Cloud Console; ledger retained.')
    else:
        if any(r.get('status') in ['submitting','running','unknown'] for r in job_records()):
            raise RuntimeError('An unresolved earlier query remains in the ledger; recover it before new queries.')
        remaining=BIGQUERY_BUDGET_USD-reserved_cost()
        # Round reserve up; BigQuery minimum and billing granularity are covered conservatively.
        needed=int(math.ceil(max(int(plan_row['estimated_bytes'])*1.05,10*1024**2)/1024**2)*1024**2)
        allowed=min(MAX_BYTES_PER_QUERY,int(max(0,remaining)/BIGQUERY_USD_PER_TIB*1024**4))
        if needed>allowed:
            return {'month':month,'status':'budget_or_per_query_limit','estimated_usd':plan_row['estimated_list_usd']}
        record=dict(job_id=jid,month=month,fingerprint=fingerprint,status='submitting',
                    budget_bytes=needed,maximum_bytes_billed=needed,created_at_utc=datetime.now(timezone.utc).isoformat())
        atomic_json(jp,record)
        try:
            job=client.query(SQL,location=LOCATION,job_id=jid,job_retry=None,
                job_config=bigquery.QueryJobConfig(query_parameters=query_params(plan_row['start'],plan_row['end_exclusive']),
                    maximum_bytes_billed=needed,use_query_cache=True))
        except Conflict:
            job=client.get_job(jid,project=GCP_PROJECT,location=LOCATION)
        except Exception:
            record['status']='unknown';atomic_json(jp,record)
            raise RuntimeError(f'{jid}: submission uncertain. Ledger reserve retained; inspect job history.') from None
    record=json.loads(jp.read_text())
    try:
        rows=job.result()
        temporary=Path(str(path)+'.tmp');count=0
        with gzip.open(temporary,'wt',encoding='utf-8') as stream:
            for row in rows:
                stream.write(json.dumps(dict(row),ensure_ascii=False,default=str)+'\n');count+=1
        temporary.replace(path)
        record.update(status='done',rows=count,budget_bytes=int(job.total_bytes_billed or 0),
                      total_bytes_billed=int(job.total_bytes_billed or 0),cache_hit=bool(job.cache_hit),path=str(path))
        atomic_json(jp,record)
        return {'month':month,'status':'downloaded','rows':count,'path':str(path)}
    except Exception:
        record['status']='unknown';atomic_json(jp,record)
        raise RuntimeError(f'{jid}: collection interrupted. Saved job can be recovered; do not start duplicate jobs.') from None

if EXECUTE_NEWS:
    statuses=[]
    for row in plan.to_dict('records'):
        status=collect_month(row);statuses.append(status);print(status)
        atomic_json(RUN/f'news_{NEWS_MODE}_collection_status.json',statuses)
        if status['status']=='budget_or_per_query_limit':
            print('Stopped before submitting the next query. Coverage remains incomplete.');break
    print('Module ledger query-compute spend/reserve: $',round(reserved_cost(),4))
else:print('Dry-run only. Set EXECUTE_NEWS=True after inspecting costs.')

## 4. Clean titles, preserve versions, and flag eligibility
Canonicalize tracking URLs conservatively; retain raw URLs. Keep the first **nonblank title version actually observed**, not a later title backdated to the URL's first discovery. Preserve other versions in raw records and export their counts.

Missing title, non-English source-language metadata, ad/commerce wording, obvious nonarticle URL paths, foreign geography and ambiguous US relevance receive separate flags. Ad flags are heuristics, not proof. US outlet location does not establish a headline's geographic subject. Keep geopolitical comparisons for eligibility review rather than automatically deleting every foreign mention.

Exact repeated titles within an outlet/month are flagged and one analytic copy retained. Identical headlines across outlets are linked, not deleted: they may reflect syndication or shared wording, not necessarily the same story. Near duplicates are a review flag, not asserted syndication. Human decisions can be merged from `news_review_decisions.csv` with `article_id`, `decision` (`Keep`/`Exclude`/`Unclear`) and `reason`. These are corpus eligibility decisions, separate from the later 300-item double annotation.

**Geographic screen:** likely US-focused, likely foreign-focused, or geographically unclear, with matched cue evidence. Mixed US/foreign titles remain in the US-cue candidate set. Foreign-focused records are held out, not deleted. Unclear headlines remain available for review and later eligibility coding, but do not silently enter the strict scoring sample. This heuristic can miss implicit US links and over-include incidental US mentions; evaluate it on human-labeled examples. Manual `Keep`/`Exclude`/`Unclear` decisions in `BASE/news_review_decisions.csv` override provisional inclusion, subject to hard text-quality checks. `Keep` should mean you checked both geography and economic relevance. Broad candidate records and held-out groups are preserved to audit selection effects.


The cleaning cell uses compiled phrase patterns, checks geography only for economic keyword matches, and maps outlet names once per distinct domain. Non-economic records are explicitly marked geographically not screened. Progress prints distinguish reading Drive files, cleaning and writing CSVs. No collection/API calls occur in this section.

In [ ]:
TRACKING_KEYS={'fbclid','gclid','mc_cid','mc_eid','guccounter','guce_referrer','guce_referrer_sig','ncid','ocid'}
def canonical_url(url):
    p=urllib.parse.urlsplit(str(url));host=(p.hostname or '').lower()
    if host.startswith('www.'):host=host[4:]
    query=urllib.parse.parse_qsl(p.query,keep_blank_values=True)
    query=[(k,v) for k,v in query if not k.lower().startswith('utm_') and k.lower() not in TRACKING_KEYS]
    return urllib.parse.urlunsplit(('https',host,p.path or '/',urllib.parse.urlencode(sorted(query)),''))

def normalize_title(title):
    s=unicodedata.normalize('NFKC',html.unescape(str(title or '')))
    return re.sub(r'\s+',' ',s).strip()

def outlet_for_domain(domain):
    hits=sources.loc[sources.domain.map(lambda d:domain==d or domain.endswith('.'+d))]
    return hits.sort_values('domain',key=lambda x:x.str.len(),ascending=False).iloc[0].outlet if len(hits) else None


# Text cues are screening evidence, not a trained geographic classifier.
US_NAMES=[ 'United States','American','Americans','Federal Reserve','Bureau of Labor Statistics',
 'Bureau of Economic Analysis','New York','NYC','California','Texas','Illinois','Houston','Chicago',
 'Los Angeles','SoCal','Southern California','New Jersey','Florida','Massachusetts','Pennsylvania',
 'Ohio','Michigan','Wisconsin','Arizona','Nevada','North Carolina','South Carolina','Tennessee',
 'Colorado','Oregon','Minnesota','Virginia','Connecticut','Maryland','Missouri','Indiana',
 'Oklahoma','Kansas','Nebraska','Iowa','Arkansas','Louisiana','Mississippi','Alabama','Kentucky',
 'New Mexico','Utah','Idaho','Montana','Wyoming','North Dakota','South Dakota','New Hampshire',
 'Vermont','Maine','Rhode Island','Delaware','West Virginia','Alaska','Hawaii','Puerto Rico']
FOREIGN_NAMES=['UK','U.K.','United Kingdom','Britain','British','England','Bank of England',
 'Finland','Finnish','Tokyo','Japan','Japanese','Slovakia','Slovak','Argentina','Argentine',
 'Eurozone','euro zone','European Central Bank','Germany','German','France','French','Italy','Italian',
 'Spain','Spanish','Portugal','Greece','Ireland','Irish','Netherlands','Dutch','Sweden','Swedish',
 'Norway','Norwegian','Denmark','Danish','Switzerland','Swiss','Austria','Austrian','Belgium',
 'Poland','Polish','Hungary','Hungarian','Czech','Romania','Romanian','China','Chinese',
 'India','Indian','Canada','Canadian','Australia','Australian','New Zealand','South Korea',
 'S.Korea','Korean','Taiwan','Taiwanese','Hong Kong','Singapore','Malaysia','Indonesia',
 'Brazil','Brazilian','Mexico','Mexican','Chile','Chilean','Colombia','Peru','Venezuela',
 'Russia','Russian','Ukraine','Ukrainian','Iran','Iranian','Israel','Israeli','Saudi Arabia',
 'Turkey','Turkish','South Africa','Nigeria','Egypt','Pakistan','Bangladesh']


def phrase_union(phrases):
    return r'(?<!\w)(?:'+'|'.join(re.escape(x) for x in sorted(set(phrases),key=len,reverse=True))+r')(?!\w)'
US_NAME_RE=re.compile(phrase_union(US_NAMES),re.I)
FOREIGN_NAME_RE=re.compile(phrase_union(FOREIGN_NAMES),re.I)
US_ACRONYM_RE=re.compile(r'(?<!\w)(?:U\.S\.?|US|USA|Fed|BLS|BEA)(?!\w)')
TERM_RE=re.compile(phrase_union([term for terms in SEARCH_TERMS.values() for term in terms]),re.I)

def cue_hits(text,phrases):
    pattern=US_NAME_RE if phrases is US_NAMES else FOREIGN_NAME_RE if phrases is FOREIGN_NAMES else re.compile(phrase_union(phrases),re.I)
    return list(dict.fromkeys(m.group(0) for m in pattern.finditer(text)))

def classify_geography(headline):
    text=normalize_title(headline)
    us=cue_hits(text,US_NAMES)
    # Case-sensitive US/Fed/BLS/BEA avoids treating the pronoun "us" or verb "fed" as US cues.
    us += [m.group(0) for m in US_ACRONYM_RE.finditer(text)]
    foreign=cue_hits(text,FOREIGN_NAMES)
    if us:
        status='Likely US-focused'
        reason='Explicit US cue; foreign context also present' if foreign else 'Explicit US geography or institution cue'
    elif foreign:
        status='Likely foreign-focused';reason='Foreign cue without an explicit US cue; held out provisionally'
    else:
        status='Geographically unclear';reason='No explicit geographic cue; outlet country is insufficient'
    return dict(geographic_relevance_status=status,geography_reason=reason,
                us_geography_cues='; '.join(dict.fromkeys(us)),foreign_geography_cues='; '.join(foreign),
                us_explicit_reference=bool(us),foreign_reference_flag=bool(foreign),
                us_linked_international_flag=bool(us and foreign))

def clean_news(raw):
    started=time.perf_counter()
    print(f'Cleaning {len(raw):,} raw records: decoding titles and URLs...',flush=True)
    d=raw.copy().fillna('')
    d['headline']=d.page_title_raw.map(normalize_title)
    unique_urls=d.url.unique()
    url_map={url:canonical_url(url) for url in unique_urls}
    d['canonical_url']=d.url.map(url_map)
    d['article_id']=d.canonical_url.map(lambda x:sha(x)[:24])
    domain_map={domain:outlet_for_domain(domain) for domain in d.domain.unique()}
    d['outlet']=d.domain.map(domain_map)
    d=d.merge(sources[['outlet','source_category','intended_geography']],on='outlet',how='left',validate='many_to_one')
    d['title_seen_at_utc']=pd.to_datetime(d.title_first_seen_raw.astype(str),format='%Y%m%d%H%M%S',utc=True,errors='coerce')
    d['publication_datetime_extracted_utc']=pd.to_datetime(d.publication_raw.astype(str),format='%Y%m%d%H%M%S',utc=True,errors='coerce')
    d['headline_missing']=d.headline.eq('')
    d['title_version_count']=d.groupby('article_id').headline.transform('nunique')
    d['gdelt_record_count']=pd.to_numeric(d.gdelt_record_count,errors='coerce').fillna(0)
    d['url_gdelt_record_count']=d.groupby('article_id').gdelt_record_count.transform('sum')
    d=d.sort_values(['article_id','headline_missing','title_seen_at_utc','headline']).drop_duplicates('article_id',keep='first').copy()
    print(f'{len(d):,} unique URLs after version selection; screening economic keywords...',flush=True)
    d['observation_type']='Archived page title / headline proxy'
    d['availability_datetime_utc']=d.title_seen_at_utc
    d['analysis_month']=d.availability_datetime_utc.dt.strftime('%Y-%m')
    d['publication_date_verified']=False
    d['publication_timing_flag']=(d.publication_datetime_extracted_utc>d.title_seen_at_utc+pd.Timedelta(days=1))
    d['language_nonenglish_flag']=~d.source_language_code.str.lower().isin(['','eng','en'])
    d['language_basis']='Selected English-language outlet plus available source-language metadata; not full validation'
    for group,pattern in SEARCH_GROUPS.items():d['keyword_'+group]=d.headline.str.contains(pattern,case=False,regex=True)
    keyword_cols=['keyword_'+k for k in SEARCH_GROUPS]
    d['economic_keyword_match']=d[keyword_cols].any(axis=1)
    d['matched_search_groups']=''
    for group in SEARCH_GROUPS:
        d['matched_search_groups']=d.matched_search_groups+np.where(d['keyword_'+group],group+';','')
    d['matched_search_groups']=d.matched_search_groups.str.rstrip(';')
    d['ad_commerce_flag']=d.headline.str.contains(r'\b(?:sponsored|advertorial|promo code|coupon|shop now|best deals|black friday deals)\b',case=False,regex=True)
    d['nonarticle_path_flag']=d.canonical_url.str.contains(r'/(?:tag|tags|category|author|search)/|/live/?(?:\?|$)',case=False,regex=True)
    # Only economic matches need geographic eligibility; no one-Series-per-row expansion.
    econ=d.economic_keyword_match
    print(f'Geographic screening for {int(econ.sum()):,} economic keyword matches...',flush=True)
    defaults={'geographic_relevance_status':'Not screened: no economic keyword match',
              'geography_reason':'Outside the keyword-defined economic pool',
              'us_geography_cues':'','foreign_geography_cues':'',
              'us_explicit_reference':False,'foreign_reference_flag':False,
              'us_linked_international_flag':False}
    for name,value in defaults.items():d[name]=value
    unique_titles=d.loc[econ,'headline'].unique()
    records={title:classify_geography(title) for title in unique_titles}
    for column in defaults:
        mapping={title:record[column] for title,record in records.items()}
        d.loc[econ,column]=d.loc[econ,'headline'].map(mapping)
    d['geography_filter_version']=GEO_FILTER_VERSION
    d['matched_search_terms']=''
    matches={title:'; '.join(dict.fromkeys(m.group(0) for m in TERM_RE.finditer(title))) for title in unique_titles}
    d.loc[econ,'matched_search_terms']=d.loc[econ,'headline'].map(matches)
    d['keyword_sense_review_flag']=d.headline.str.contains(r'\b(?:balloons?|inflatable)\b',case=False,regex=True)&d.headline.str.contains(r'\b(?:dress|dresses|fashion|party|parties)\b',case=False,regex=True)
    d['normalized_headline']=d.headline.str.casefold().str.replace(r'[^\w\s]','',regex=True).str.replace(r'\s+',' ',regex=True).str.strip()
    d['headline_cluster']=d.normalized_headline.map(lambda x:sha(x)[:24] if x else '')
    d=d.sort_values(['title_seen_at_utc','article_id'])
    d['within_outlet_month_repeat']=d.duplicated(['outlet','analysis_month','normalized_headline'],keep='first') & d.headline.ne('')
    d['same_title_outlet_count']=d.groupby(['analysis_month','headline_cluster']).outlet.transform('nunique')
    d['possible_syndication_flag']=d.same_title_outlet_count.gt(1)&d.headline.ne('')
    d['engagement_available']=False;d['actual_exposure_available']=False
    d['eligibility_status']='Candidate; final economic/US eligibility requires coding'
    d['review_decision']='';d['review_reason']=''
    print(f'Cleaning complete in {time.perf_counter()-started:.1f}s; writing audit files next...',flush=True)
    return d

raw_frames=[];month_status=[]
for month in all_months:
    fingerprint,path=query_identity(month)
    exists=path.exists()
    month_status.append({'month':month,'download_completed':exists,'raw_path':str(path)})
    if exists:
        print(f'Reading saved headlines: {month} ({path.stat().st_size/1024**2:.1f} MiB compressed)...',flush=True)
        with gzip.open(path,'rt',encoding='utf-8') as stream:
            rows=[json.loads(line) for line in stream if line.strip()]
        if rows:raw_frames.append(pd.DataFrame(rows))
        print(f'  Loaded {len(rows):,} records.',flush=True)
month_status=pd.DataFrame(month_status)
month_status.to_csv(RUN/'news_month_download_status.csv',index=False)
if raw_frames:
    raw_news=pd.concat(raw_frames,ignore_index=True)
    news=clean_news(raw_news)
    decisions_path=BASE/'news_review_decisions.csv'
    if decisions_path.exists():
        decisions=pd.read_csv(decisions_path,dtype=str).fillna('')
        assert not decisions.article_id.duplicated().any()
        if not decisions.decision.isin(['Keep','Exclude','Unclear','']).all():raise ValueError('Invalid review decision')
        lookup=decisions.set_index('article_id')
        news['review_decision']=news.article_id.map(lookup.decision).fillna('')
        news['review_reason']=news.article_id.map(lookup.reason).fillna('')
    print('Writing retrieval audit and eligibility queues to Drive...',flush=True)
    news.to_csv(RUN/'news_retrieval_audit.csv',index=False)
    hard_ok=~news.headline_missing&~news.language_nonenglish_flag&news.title_seen_at_utc.notna()
    flagged=news.ad_commerce_flag|news.nonarticle_path_flag|news.publication_timing_flag
    keep=hard_ok&~news.within_outlet_month_repeat&~news.review_decision.isin(['Exclude','Unclear'])&(~flagged|news.review_decision.eq('Keep'))
    economic_pool=news.loc[keep&news.economic_keyword_match].copy()
    economic_pool.to_csv(RUN/'news_economic_candidates_all_geographies.csv',index=False)
    us_screen=(economic_pool.geographic_relevance_status.eq('Likely US-focused')&~economic_pool.keyword_sense_review_flag)|economic_pool.review_decision.eq('Keep')
    candidates=economic_pool.loc[us_screen].copy()
    candidates['eligibility_status']='Strict US-cue candidate or manual Keep; final eligibility still requires validation'
    candidates.to_csv(RUN/'news_headline_candidates.csv',index=False)
    unclear=economic_pool.loc[economic_pool.geographic_relevance_status.eq('Geographically unclear')|economic_pool.keyword_sense_review_flag].copy()
    unclear.to_csv(RUN/'news_geography_unclear_review.csv',index=False)
    foreign=economic_pool.loc[economic_pool.geographic_relevance_status.eq('Likely foreign-focused')&~economic_pool.review_decision.eq('Keep')].copy()
    foreign.to_csv(RUN/'news_foreign_focused_held_out.csv',index=False)
    geography_counts=economic_pool.groupby(['geographic_relevance_status','outlet']).size().rename('headlines').reset_index()
    geography_counts.to_csv(RUN/'news_geography_counts.csv',index=False)
    display(economic_pool.geographic_relevance_status.value_counts().rename('Economic candidates by geography'))
    print('Strict US working candidates:',len(candidates),'| unclear/sense review:',len(unclear),'| foreign held out:',len(foreign))
    # Quality spotcheck includes flagged candidates, plus five per outlet/available year.
    news['year']=news.title_seen_at_utc.dt.year
    review_ids=set(news.loc[flagged&news.economic_keyword_match,'article_id'])
    for _,g in news.loc[hard_ok&news.economic_keyword_match].groupby(['outlet','year']):
        review_ids.update(g.assign(order=g.article_id.map(lambda s:sha(SAMPLING_SEED+s))).sort_values('order').head(5).article_id)
    news.loc[news.article_id.isin(review_ids)].to_csv(RUN/'news_quality_review_queue.csv',index=False)
    print('Unique URLs:',len(news),'| economic headline candidates:',len(candidates))
else:
    news=pd.DataFrame();candidates=pd.DataFrame()
    print('No downloaded news yet. Inspect dry-run costs, then run the pilot.')

## 5. Coverage, deterministic scoring sample and fixed-panel readiness
Retain **all eligible candidates** as the broad collection. For affordable later scoring, select at most 50 candidates per outlet/month by a stable seeded hash. Fewer articles remain fewer; never duplicate rows to fill quotas. This is approximately balanced **within the observed archive**, not balanced by sentiment. Candidate review may change which items are eligible; freeze the final scoring sample before interpreting results.

Report all-source, equal-outlet, equal-category and fixed-source-panel results later. Quota sampling changes selection/salience denominators: use the unsampled candidate counts for coverage volume and use transparent weights for sampled label shares. A common panel must have adequate coverage in every complete month. No entire missing source-month is filled as neutral or zero sentiment. Pilot-only or budget-stopped data are not regression-ready.

In [ ]:
if not news.empty:
    grid=pd.MultiIndex.from_product([sources.outlet,all_months],names=['outlet','analysis_month']).to_frame(index=False)
    counts=news.groupby(['outlet','analysis_month']).agg(retrieved_urls=('article_id','size'),
        missing_titles=('headline_missing','sum'),economic_keyword_candidates=('economic_keyword_match','sum')).reset_index()
    usable=candidates.groupby(['outlet','analysis_month']).size().rename('working_candidates').reset_index()
    coverage=grid.merge(counts,how='left').merge(usable,how='left').merge(sources[['outlet','source_category']],how='left')
    for col in ['retrieved_urls','missing_titles','economic_keyword_candidates','working_candidates']:coverage[col]=coverage[col].fillna(0).astype(int)
    coverage=coverage.merge(month_status[['month','download_completed']],left_on='analysis_month',right_on='month',how='left')
    last_full=(pd.Timestamp(NEWS_END_EXCLUSIVE).to_period('M')-1).strftime('%Y-%m')
    # A historical mid-month end also makes its endpoint month incomplete.
    coverage['complete_calendar_month']=(coverage.analysis_month<=last_full)&(coverage.analysis_month>=(pd.Timestamp(NEWS_START).to_period('M') + (0 if pd.Timestamp(NEWS_START).day==1 else 1)).strftime('%Y-%m'))
    coverage['coverage_status']=np.where(~coverage.download_completed,'Not downloaded',np.where(coverage.working_candidates.eq(0),'Downloaded; zero usable keyword candidates','Downloaded; candidates available'))
    coverage.to_csv(RUN/'news_outlet_month_coverage.csv',index=False)
    candidates['sample_order']=candidates.article_id.map(lambda x:sha(SAMPLING_SEED+x))
    sample=candidates.sort_values('sample_order').groupby(['outlet','analysis_month'],group_keys=False).head(MAX_PER_OUTLET_MONTH).copy()
    population=candidates.groupby(['outlet','analysis_month']).size()
    sampled=sample.groupby(['outlet','analysis_month']).size()
    sample['candidate_stratum_n']=[population.loc[(r.outlet,r.analysis_month)] for r in sample.itertuples()]
    sample['sample_stratum_n']=[sampled.loc[(r.outlet,r.analysis_month)] for r in sample.itertuples()]
    sample['within_stratum_sampling_weight']=sample.candidate_stratum_n/sample.sample_stratum_n
    sample['evaluation_split']='Unassigned; preserve separate development/held-out manifest in labeling module'
    sample.to_csv(RUN/'news_headlines_for_labeling.csv',index=False)
    complete=coverage.loc[coverage.complete_calendar_month]
    panel=complete.groupby('outlet').agg(months=('analysis_month','size'),downloaded_months=('download_completed','sum'),
        minimum_candidates=('working_candidates','min'))
    panel['fixed_panel_eligible']=panel.months.eq(panel.downloaded_months)&panel.minimum_candidates.ge(MIN_PANEL_HEADLINES)
    panel.to_csv(RUN/'fixed_panel_readiness.csv')
    ready=bool(month_status.download_completed.all())
    atomic_json(RUN/'news_readiness.json',{'all_requested_months_downloaded':ready,
       'news_mode':NEWS_MODE,'candidate_count':len(candidates),'scoring_sample_count':len(sample),
       'status':'Collection complete; eligibility validation and labeling still required' if ready else 'INCOMPLETE: pilot or budget-stopped; no historical regression yet'})
    display(coverage.pivot(index='outlet',columns='analysis_month',values='working_candidates'))
    display(panel)
    print('Scoring sample:',len(sample),'| all requested months downloaded:',ready)

## 6. Review the news relevance FIRST, then freeze shared terms
Inspect `news_term_relevance_review.csv` and the displayed headlines. The review includes up to eight matching titles per source/topic plus up to three keyword-negative titles per source/month to look for missed economic coverage. Keyword groups are screening rules, not human truth. Check economic meaning, US relevance, advertisements and page-title quality.

If terms are noisy or miss useful coverage, edit `SEARCH_TERMS`, change `TERMS_VERSION`, and rerun configuration, term definitions and cleaning. **Reuse the same downloaded news**; no new scan is needed for vocabulary edits. Do this before inspecting sentiment/correlation results. No automatic term optimization is performed.

Once satisfied, set `NEWS_TERMS_REVIEWED=True` in configuration and rerun through this cell. This freezes a terms manifest for X, including the reviewed source-month coverage. It is a deliberate relevance check; it is not the later held-out human-label accuracy evaluation. You can review the small historical news pilot before committing to the full news download.

The default preview now shows strict US-cue candidates. The same CSV also contains small geography-audit samples of held-out foreign and unclear records, so you can check false exclusions. Inspect `news_geography_unclear_review.csv` for the full unresolved queue. The top preview alone does not represent all screening decisions.

In [ ]:
review_manifest=RUN/'reviewed_news_terms.json'
review_ready=False
if not news.empty and not candidates.empty:
    review_parts=[]
    for group in SEARCH_TERMS:
        subset=candidates.loc[candidates['keyword_'+group]&~candidates.headline_missing].copy()
        subset['review_topic']=group
        subset['review_sort']=subset.article_id.map(lambda v:sha(SAMPLING_SEED+group+v))
        review_parts.append(subset.sort_values('review_sort').groupby('outlet',group_keys=False).head(8))
    negatives=news.loc[~news.economic_keyword_match&~news.headline_missing].copy()
    negatives['review_topic']='Keyword-negative check for missed economic headlines'
    negatives['review_sort']=negatives.article_id.map(lambda v:sha(SAMPLING_SEED+'negative'+v))
    for status,label in [('Likely foreign-focused','Geography audit: likely foreign'),('Geographically unclear','Geography audit: unclear')]:
        subset=news.loc[news.economic_keyword_match&news.geographic_relevance_status.eq(status)].copy()
        subset['review_topic']=label
        subset['review_sort']=subset.article_id.map(lambda v:sha(SAMPLING_SEED+label+v))
        review_parts.append(subset.sort_values('review_sort').groupby('outlet',group_keys=False).head(3))
    review_parts.append(negatives.sort_values('review_sort').groupby(['outlet','analysis_month'],group_keys=False).head(3))
    relevance_review=pd.concat(review_parts,ignore_index=True)
    cols=['article_id','outlet','analysis_month','headline','review_topic','matched_search_groups',
          'matched_search_terms','geographic_relevance_status','geography_reason','us_geography_cues',
          'foreign_geography_cues','keyword_sense_review_flag','ad_commerce_flag','url']
    relevance_review=relevance_review[cols].drop_duplicates(['article_id','review_topic'])
    relevance_review['review_relevance']='';relevance_review['review_note']=''
    review_csv=RUN/'news_term_relevance_review.csv'
    if review_csv.exists():
        previous_review=pd.read_csv(review_csv,dtype=str).fillna('').drop_duplicates(['article_id','review_topic'])
        previous_review=previous_review.set_index(['article_id','review_topic'])
        keys=pd.MultiIndex.from_frame(relevance_review[['article_id','review_topic']])
        for column in ['review_relevance','review_note']:
            if column in previous_review:
                relevance_review[column]=previous_review[column].reindex(keys).fillna('').to_numpy()
    relevance_review.to_csv(review_csv,index=False)
    pd.set_option('display.max_colwidth',None)
    display(relevance_review[['outlet','headline','review_topic','matched_search_terms','geographic_relevance_status']].head(40))
    print('Review queues retain full headlines and geographic evidence.')
    print('Candidate counts by keyword group (items can match multiple groups):')
    print(candidates[['keyword_'+g for g in SEARCH_TERMS]].sum())
    if NEWS_TERMS_REVIEWED:
        manifest={'terms':SEARCH_TERMS,'terms_hash':TERMS_HASH,'terms_version':TERMS_VERSION,'geo_filter_version':GEO_FILTER_VERSION,
                  'reviewed_at_utc':datetime.now(timezone.utc).isoformat(),
                  'review_basis':'User inspected news relevance examples before X collection',
                  'news_months_downloaded':month_status.loc[month_status.download_completed,'month'].tolist(),
                  'news_candidate_count':len(candidates),'not_sentiment_optimized':True}
        if review_manifest.exists() and (json.loads(review_manifest.read_text())['terms_hash']!=TERMS_HASH or json.loads(review_manifest.read_text()).get('geo_filter_version')!=GEO_FILTER_VERSION):
            raise ValueError('Terms differ from frozen review. Change TERMS_VERSION and review new terms.')
        if not review_manifest.exists():atomic_json(review_manifest,manifest)
        review_ready=True
        print('Terms frozen for historical X collection.')
    else:
        print('News review stage. Inspect headlines, then set NEWS_TERMS_REVIEWED=True to enable X planning.')
else:
    print('Collect the news pilot first. X planning stays disabled until news examples are reviewed.')

## 7. Historical X collection with the reviewed news vocabulary
Provider: **twitterapi.io**, an independent third-party service. The period now matches the news history: **October 2019 onward**. The provider advertises access back to 2006; a historical pilot tests the actual queries and access in early, middle and recent years before full collection.

Target **600 returned posts per complete month**: 30 pre-specified windows × about 20 results each. Windows span each month, rotate the six economic query families, and use deterministic time-of-day jitter. Each query family uses the SAME words/phrases frozen after the news relevance review. X additionally requires an explicit US cue, English and attempted repost exclusion. No engagement threshold or political-name condition is used.

Each window requests one page of `Latest` matches from a two-hour interval. This is a stratified, capped convenience sample, not random sampling of all X. Sparse windows yield fewer posts; never fill shortfalls by selecting favorable results. Duplicates, out-of-window records and later eligibility checks can further reduce usable counts. Partial-month targets are proportional to the number of scheduled windows completed by the cutoff.

**Do not interpret the imposed topic-query balance as natural relative topic attention on X.** Likewise, about 600 posts/month does not establish representativeness or eliminate author clustering.

At $0.15/1,000 returned posts, 84 × 600 = 50,400 complete-month results would cost about $7.56 for collection. Partial October, pilot requests and minimum/duplicate-result charges are additional. A **$10 cumulative local budget** covers the collector; labeling is separate. Unknown request charges stop execution. Paid requests start only when both `NEWS_TERMS_REVIEWED=True` and `EXECUTE_X=True`; use `X_MODE='pilot'` first, then `'history'`. Historical execution may take hours because calls are spaced for rate limits. Checkpointing permits restart.

In [ ]:
X_US_CUE='("United States" OR "U.S." OR "US economy" OR "American economy" OR "Federal Reserve")'

def build_x_groups(terms):
    # Quoting multiword terms preserves the exact shared phrase vocabulary.
    def token(term):
        term=term.replace('"','').strip()
        return '"'+term+'"' if ' ' in term else term
    return {group:'('+' OR '.join(token(term) for term in words)+')' for group,words in terms.items()}
X_QUERY_GROUPS=build_x_groups(SEARCH_TERMS)

def x_windows(terms, start_date=X_START, end_date=X_END_EXCLUSIVE):
    rows=[];start=pd.Timestamp(start_date,tz='UTC');end=pd.Timestamp(end_date,tz='UTC')
    n_windows=math.ceil(X_TARGET_PER_COMPLETE_MONTH/X_EXPECTED_RESULTS_PER_PAGE)
    groups=list(terms);query_groups=build_x_groups(terms)
    for period in pd.period_range(start.tz_localize(None).to_period('M'),(end-pd.Timedelta(seconds=1)).tz_localize(None).to_period('M'),freq='M'):
        month_start=period.to_timestamp().tz_localize('UTC')
        month_end=(period+1).to_timestamp().tz_localize('UTC')
        month_seconds=int((month_end-month_start).total_seconds())
        rotation=int(sha(SAMPLING_SEED+str(period))[:8],16)%len(groups)
        for slot in range(n_windows):
            left=month_start+pd.Timedelta(seconds=month_seconds*slot//n_windows)
            right=month_start+pd.Timedelta(seconds=month_seconds*(slot+1)//n_windows)
            width=min(int(X_WINDOW_HOURS*3600),int((right-left).total_seconds()))
            room=max(0,int((right-left).total_seconds())-width)
            jitter=int(sha(SAMPLING_SEED+str(period)+str(slot))[:12],16)%(room+1)
            lo=left+pd.Timedelta(seconds=jitter);hi=lo+pd.Timedelta(seconds=width)
            if lo<start or hi>end:continue
            group=groups[(slot+rotation)%len(groups)]
            query=(f'{query_groups[group]} {X_US_CUE} lang:en -filter:retweets '
                   f'since_time:{int(lo.timestamp())} until_time:{int(hi.timestamp())}')
            rows.append(dict(date=str(lo.date()),month=str(period),group=group,query=query,
                start_utc=lo.isoformat(),end_utc=hi.isoformat(),slot=slot,
                sample_window_id=sha(str(period)+str(slot)+query)[:24],terms_hash=TERMS_HASH,
                complete_calendar_month=month_start>=start and month_end<=end))
    return pd.DataFrame(rows,columns=['date','month','group','query','start_utc','end_utc','slot',
                                      'sample_window_id','terms_hash','complete_calendar_month'])

xplan=pd.DataFrame();x_selected=pd.DataFrame()
if review_ready:
    assert json.loads(review_manifest.read_text())['terms_hash']==TERMS_HASH
    xplan=x_windows(SEARCH_TERMS)
    if X_MODE=='pilot' and not xplan.empty:
        months=sorted(xplan.month.unique())
        # Early, middle (around 2022), and latest complete month; nine requests by default.
        mid=min(months,key=lambda m:abs((pd.Period(m,'M')-pd.Period('2022-07','M')).n))
        complete=sorted(xplan.loc[xplan.complete_calendar_month,'month'].unique())
        recent=complete[-1] if complete else months[-1]
        chosen=sorted(set([months[0],mid,recent]));pieces=[]
        for month in chosen:
            group=xplan.loc[xplan.month.eq(month)].sort_values('start_utc')
            positions=sorted(set([0,len(group)//2,len(group)-1]))
            pieces.append(group.iloc[positions])
        x_selected=pd.concat(pieces,ignore_index=True)
    elif X_MODE=='history':x_selected=xplan.copy()
    else:
        if X_MODE!='pilot':raise ValueError("X_MODE must be 'pilot' or 'history'.")
        x_selected=xplan.copy()
    xplan.to_csv(RUN/'x_all_query_windows.csv',index=False)
    x_selected.to_csv(RUN/'x_selected_query_windows.csv',index=False)
    atomic_json(RUN/'x_query_terms.json',{'groups':X_QUERY_GROUPS,'shared_terms':SEARCH_TERMS,
                'terms_hash':TERMS_HASH,'us_cue':X_US_CUE,'window_hours':X_WINDOW_HOURS,
                'target_per_complete_month':X_TARGET_PER_COMPLETE_MONTH,'query_type':'Latest'})
    if not xplan.empty:
        schedule=xplan.groupby('month').agg(windows=('query','size'),complete_calendar_month=('complete_calendar_month','all')).reset_index()
        schedule['expected_returned_posts']=schedule.windows*X_EXPECTED_RESULTS_PER_PAGE
        schedule['estimated_collection_usd']=schedule.expected_returned_posts*X_USD_PER_TWEET
        schedule.to_csv(RUN/'x_monthly_sampling_plan.csv',index=False)
        display(schedule)
        print('Selected requests:',len(x_selected),'| expected collection cost: $',
              round(len(x_selected)*X_EXPECTED_RESULTS_PER_PAGE*X_USD_PER_TWEET,3))
        print('Full scheduled history estimate (before pilot extras): $',round(schedule.estimated_collection_usd.sum(),3))
        print('Targets may be missed in sparse windows; no automatic filling.')
else:
    print('X disabled until news relevance review. No tweet requests or key prompt.')

# Query text is reproducibility metadata; keys are stored only in memory.
def x_cost_reserved():
    return sum(json.loads(p.read_text()).get('budget_usd',0) for p in XRAW.glob('*.meta.json'))

def collect_x_window(window,key):
    cursor='';seen_cursors=set()
    for page in range(X_MAX_PAGES_PER_WINDOW):
        request_identity=json.dumps({'endpoint':'advanced_search','query':window['query'],'queryType':'Latest','cursor':cursor},sort_keys=True)
        token=sha(request_identity);path=XRAW/(token+'.json');meta_path=XRAW/(token+'.meta.json')
        if path.exists():
            payload=json.loads(path.read_text())
        else:
            if meta_path.exists():raise RuntimeError('Unresolved X request reserve. Check provider billing before a paid retry.')
            reserve=max(X_MIN_REQUEST_USD,X_RESERVE_RESULTS_PER_REQUEST*X_USD_PER_TWEET)
            if x_cost_reserved()+reserve>X_BUDGET_USD:return 'budget_stop'
            meta={'provider':'twitterapi.io','platform':'X','query':window['query'],'group':window['group'],'date':window['date'],
                  'cursor':cursor,'page':page,**{k:window[k] for k in ['month','start_utc','end_utc','slot','sample_window_id','terms_hash','complete_calendar_month']},'budget_usd':reserve,'status':'requesting','retrieved_at_utc':datetime.now(timezone.utc).isoformat()}
            atomic_json(meta_path,meta)
            params={'query':window['query'],'queryType':'Latest','cursor':cursor}
            url='https://api.twitterapi.io/twitter/tweet/advanced_search?'+urllib.parse.urlencode(params)
            request=urllib.request.Request(url,headers={'X-API-Key':key,'User-Agent':'Academic economic headlines study'})
            try:
                with urllib.request.urlopen(request,timeout=60) as response:payload=json.load(response)
                if not isinstance(payload.get('tweets'),list):raise ValueError('Unexpected response schema')
                atomic_json(path,payload)
                charge=max(X_MIN_REQUEST_USD,len(payload['tweets'])*X_USD_PER_TWEET)
                meta.update(status='done',budget_usd=charge,returned_posts=len(payload['tweets']),
                            has_next_page=bool(payload.get('has_next_page')),next_cursor=payload.get('next_cursor',''))
                atomic_json(meta_path,meta)
                if len(payload['tweets'])>X_RESERVE_RESULTS_PER_REQUEST:
                    raise RuntimeError('Provider returned more than reserved; collection stopped. Inspect billing.')
            except Exception:
                raise RuntimeError('X request failed or uncertain. Reserve retained; inspect saved metadata/provider billing before retrying.') from None
            time.sleep(X_REQUEST_SPACING_SECONDS)
        new_cursor=payload.get('next_cursor') or ''
        if not payload.get('has_next_page') or not new_cursor:return 'finished'
        if new_cursor in seen_cursors:return 'repeated_cursor_stop'
        seen_cursors.add(new_cursor);cursor=new_cursor
    return 'page_cap_reached'

if EXECUTE_X and review_ready:
    key=os.environ.get('TWITTERAPI_IO_KEY') or getpass.getpass('twitterapi.io API key: ')
    xstatuses=[]
    for window in x_selected.to_dict('records'):
        status=collect_x_window(window,key);xstatuses.append({**window,'status':status})
        print(window['date'],window['group'],status)
        atomic_json(RUN/'x_collection_status.json',xstatuses)
        if status=='budget_stop':break
    del key
else:print('X calls disabled. Review news terms, then enable EXECUTE_X and run the historical pilot first.')

## 8. Preserve posts, duplicates, engagement and automation indicators
Flatten tweet responses without changing post text. Preserve IDs, dates, author IDs, source queries, retrieval time, replies/quotes/reposts when identified, engagement counts and available author location. Flag repeated text, frequent posting within this sample and low lexical diversity for review. These heuristics **do not establish bot status**. Unknown repost/ad fields remain unknown; lack of metadata is not proof of absence.

Engagement is a retrieval-time snapshot and can grow after posting. Historical likes/views downloaded today were not necessarily available at an old forecast origin. Do not use them as real-time predictors without contemporaneous snapshots. Likes/reposts/views are imperfect attention measures; they do not identify what Michigan respondents encountered.

In [ ]:
def flatten_x(payload,meta):
    rows=[]
    for post in payload.get('tweets',[]):
        author=post.get('author') or {}
        is_rt=post.get('isRetweet')
        if is_rt is None and 'retweeted_tweet' in post:is_rt=bool(post.get('retweeted_tweet'))
        rows.append(dict(post_id=str(post.get('id') or ''),text=post.get('text') or '',url=post.get('url') or '',
            created_at=post.get('createdAt'),author_id=str(author.get('id') or ''),author_username=author.get('userName') or '',
            author_location=author.get('location') or '',author_followers=author.get('followers'),
            is_reply=post.get('isReply'),is_quote=post.get('isQuote'),is_retweet=is_rt,
            like_count=post.get('likeCount'),repost_count=post.get('retweetCount'),reply_count=post.get('replyCount'),
            quote_count=post.get('quoteCount'),view_count=post.get('viewCount'),
            query=meta['query'],query_group=meta['group'],sample_day=meta['date'],
            sample_window_id=meta.get('sample_window_id',''),sample_month=meta.get('month',''),
            sample_window_start_utc=meta.get('start_utc'),sample_window_end_utc=meta.get('end_utc'),
            terms_hash=meta.get('terms_hash',''),
            retrieved_at_utc=meta['retrieved_at_utc'],window_truncated=bool(meta.get('has_next_page')),
            provider='twitterapi.io',platform='X',observation_type='Post text',actual_exposure_available=False))
    return rows

xrows=[]
for path in XRAW.glob('*.json'):
    if path.name.endswith('.meta.json'):continue
    meta_path=path.with_suffix('.meta.json')
    if not meta_path.exists():continue
    meta=json.loads(meta_path.read_text())
    if X_START<=meta.get('date','')<X_END_EXCLUSIVE and meta.get('terms_hash')==TERMS_HASH:
        xrows.extend(flatten_x(json.loads(path.read_text()),meta))
if xrows:
    xall=pd.DataFrame(xrows)
    xall.to_csv(RUN/'x_all_returned_posts_with_query_matches.csv',index=False)
    x=xall.loc[xall.post_id.ne('')].sort_values('retrieved_at_utc').drop_duplicates('post_id').copy()
    x['created_at_utc']=pd.to_datetime(x.created_at,utc=True,errors='coerce')
    x['analysis_month']=x.created_at_utc.dt.strftime('%Y-%m')
    x['normalized_text']=x.text.map(normalize_title).str.casefold()
    x['duplicate_text_flag']=x.duplicated('normalized_text',keep=False)
    x['author_posts_in_sample_day']=x.groupby(['author_id','sample_day']).post_id.transform('size')
    x['high_frequency_in_sample_flag']=x.author_posts_in_sample_day.ge(20)&x.author_id.ne('')
    x['suspected_automation_review_flag']=x.high_frequency_in_sample_flag&x.duplicate_text_flag
    x['automation_status']='Unknown; heuristic review flags only'
    x['ad_review_flag']=x.text.str.contains(r'\b(?:sponsored|#ad|promo code|affiliate link)\b',case=False,regex=True)
    x['geographic_status']='US-keyword query; author location is self-reported; not respondent exposure'
    x['window_start_utc']=pd.to_datetime(x.sample_window_start_utc,utc=True,errors='coerce')
    x['window_end_utc']=pd.to_datetime(x.sample_window_end_utc,utc=True,errors='coerce')
    x['out_of_requested_window_flag']=~(x.created_at_utc.ge(x.window_start_utc)&x.created_at_utc.lt(x.window_end_utc))
    x['eligible_for_review']=x.created_at_utc.notna()&x.text.str.strip().ne('')&~x.is_retweet.eq(True)&~x.out_of_requested_window_flag
    x.to_csv(RUN/'x_posts_cleaned_with_flags.csv',index=False)
    x.loc[x.eligible_for_review].to_csv(RUN/'x_posts_for_labeling.csv',index=False)
    x.groupby('analysis_month').agg(posts=('post_id','size'),unique_authors=('author_id','nunique'),
        duplicate_text=('duplicate_text_flag','sum'),automation_review=('suspected_automation_review_flag','sum')).to_csv(RUN/'x_monthly_coverage.csv')
    print('Unique X posts:',len(x),'| repeated returned matches:',len(xall)-len(x))
    if not xplan.empty:
        completed=[]
        for metadata_path in XRAW.glob('*.meta.json'):
            record=json.loads(metadata_path.read_text())
            if record.get('terms_hash')==TERMS_HASH and record.get('status')=='done':completed.append(record.get('sample_window_id'))
        month_report=xplan.groupby('month').agg(planned_windows=('sample_window_id','size'),complete_calendar_month=('complete_calendar_month','all')).reset_index()
        done=xplan.loc[xplan.sample_window_id.isin(completed)].groupby('month').size()
        month_report['completed_windows']=month_report.month.map(done).fillna(0).astype(int)
        unique_counts=x.loc[x.eligible_for_review].groupby('analysis_month').post_id.nunique()
        month_report['unique_posts_for_eligibility_review']=month_report.month.map(unique_counts).fillna(0).astype(int)
        month_report['scheduled_target']=month_report.planned_windows*X_EXPECTED_RESULTS_PER_PAGE
        month_report['target_shortfall']=(month_report.scheduled_target-month_report.unique_posts_for_eligibility_review).clip(lower=0)
        month_report['all_scheduled_windows_collected']=month_report.planned_windows.eq(month_report.completed_windows)
        month_report.to_csv(RUN/'x_monthly_collection_readiness.csv',index=False)
        display(month_report)
else:print('No X posts collected. The social-platform requirement is not yet satisfied.')

## 9. Reproduce, update and report limitations
**Reproduce:** rerun against saved raw gzip/JSON files and frozen configuration. Query SQL, search terms, domain registry, job IDs, costs, retrieval timestamps, checksums and coverage tables are saved. BigQuery metadata/schema and authentication are still needed for live dry runs. Raw cached responses permit cleaning without another paid query.

**Update:** retain the same source list and search rules, use a new SNAPSHOT_ID, and extend the exclusive end date. Closed-month query fingerprints reuse raw files; an extended partial month receives a new fingerprint. Rebuilding a snapshot selects the correct query interval per month, preventing duplicate concatenation. Old snapshots remain reviewable. X saved query/cursor pages are reused; retries are never silently paid. The persistent dollar ceiling is cumulative across snapshots of this module. Increase it deliberately if authorizing more collection; it is not reset by changing SNAPSHOT_ID.

**Report:** source selection and geographies; keyword expressions; date windows and UTC convention; page-title proxy versus full post text; missing titles and source-months; translation/geographic uncertainty; archive and API retrieval restrictions; syndicated/repeated items; ad/automation heuristics; query/page caps; partial months; unknown audience; and differences between source supply and engagement. Neither measures actual respondent exposure.

The query captures GDELT discoveries, not every article an outlet published. A zero count may be missing monitoring or titles. Extracted publication dates remain unverified. Monthly publication-aligned analysis and release-event comparisons need a verified-timing subset; the default date is when a title was observed by GDELT. Outlet-specific page-title practices and coverage changes may create artificial trends.

For X, paid results cannot recover deleted/private/unindexed posts. Historical access and result ordering can change. A fixed query/day cap is a convenience sample and may overrepresent late-day activity. Do not equate keyword-specific social posts with a representative US population sentiment measure.

**Next module:** freeze a headline-specific coding guide; select the same 300 stratified items for two independent annotators (including separately identified X posts); preserve a development/held-out split and duplicate clusters. No annotations or sentiment predictions have been generated here. News headlines and X posts require separate validation. Factual calibration needs official release evidence and publication timing; a headline alone cannot establish omitted context.

A nominal 600-post complete-month target is 30 time windows, not a guarantee. Report shortfalls and all downloaded/scheduled windows. Do not interpret imposed query-topic quotas as natural topic prevalence. A relevance term revision creates a new TERMS_VERSION; the same downloaded news can be re-screened without new queries. Shared terms are frozen before tweets and before outcome-based term selection.

In [ ]:
study_status={'geo_filter_version':GEO_FILTER_VERSION,'terms_hash':TERMS_HASH,'news_terms_reviewed':review_ready,
    'x_target_per_complete_month':X_TARGET_PER_COMPLETE_MONTH,
    'news_collection_complete':bool(month_status.download_completed.all()),
    'news_observation_type':'GDELT page title used as headline proxy',
    'social_provider':'twitterapi.io; independent third-party access to public X posts',
    'social_posts_collected':len(x) if xrows else 0,
    'social_returned_query_matches':len(xrows),
    'engagement_is_exposure':False,'respondent_exposure_observed':False,
    'numerical_audience_estimates_available':False,
    'bigquery_compute_spend_or_reserve_usd':reserved_cost(),
    'twitterapi_io_spend_or_reserve_usd':x_cost_reserved(),
    'publication_timestamps_verified':False,
    'analysis_status':'Collection and eligibility screening; no sentiment or causal conclusions'}
atomic_json(RUN/'study_status.json',study_status)
import importlib.metadata
package_versions={name:importlib.metadata.version(name) for name in ['pandas','numpy','google-cloud-bigquery']}
manifest={'packages':package_versions,'python':sys.version,'config':COLLECTION_CONFIG,'generated_at_utc':datetime.now(timezone.utc).isoformat(),'files':[]}
for root in [RUN,RAWSET,XRAW,JOBDIR]:
    for path in sorted(root.glob('*')):
        if path.is_file() and path.name!='manifest.json':
            manifest['files'].append({'path':str(path.relative_to(BASE)),'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'bytes':path.stat().st_size})
atomic_json(RUN/'manifest.json',manifest)
print('Snapshot outputs:',RUN)
print(json.dumps(study_status,indent=2))

### Primary documentation
- [GDELT titles introduced September 22, 2019](https://blog.gdeltproject.org/gkg-2-0-now-includes-page-titles/)
- [GDELT title extraction and HTML escaping](https://blog.gdeltproject.org/unescaping-article-titles-in-the-gkg-2-0/)
- [GDELT extracted publication metadata](https://blog.gdeltproject.org/new-gkg-2-0-article-metadata-fields/)
- [BigQuery dry runs and maximum bytes billed](https://docs.cloud.google.com/bigquery/docs/best-practices-costs)
- [BigQuery pricing](https://cloud.google.com/bigquery/pricing)
- [twitterapi.io advanced search](https://docs.twitterapi.io/api-reference/endpoint/tweet_advanced_search)
- [Provider query parameters and pagination examples](https://twitterapi.io/blog/twitter-advanced-search-guide)
- [twitterapi.io pricing](https://twitterapi.io/pricing)

The notebook has been syntax checked and tested using synthetic records. Authenticated BigQuery queries and paid twitterapi.io requests have not been executed during creation. Your historical pilot and dry run validate live schemas, access and cost.